# Arena 3D Reconstruction with Gaussian Splatting

Reconstruct a 10-15m arena from 91 photos using COLMAP + 3D Gaussian Splatting.

**Camera:** Google Pixel 10 Pro XL (24mm equiv., f/1.68)  
**COLMAP model:** SIMPLE_RADIAL (fx=1398, k1=0.0065)  
**Images:** 91 resized to 1920px

**Design:** Each cell runs 2-10 minutes, saves intermediate results to Google Drive, and skips itself if a checkpoint already exists. You can resume from any point after a crash by re-running the cell that failed.

---
**Workflow:**
1. Mount Drive + install deps (~3 min, idempotent)
2. Download images (~2 min, idempotent)
3. COLMAP: feature extraction → matching → mapping (3 cells, ~5+5+10 min, idempotent)
4. Model merging (~2 min, idempotent)
5. Convert data (~1 min)
6. Training in segments (each ~10 min, checkpointed)
7. Export + Unity validation (~1 min)
---

In [ ]:
#@title === 1. Mount Google Drive (for checkpoint saving) ===
import os
from google.colab import drive

DRIVE_PATH = "/content/drive/MyDrive/arena_3dgs"
drive.mount('/content/drive')
os.makedirs(DRIVE_PATH, exist_ok=True)
print(f"Checkpoints will be saved to: {DRIVE_PATH}")

# ── Helper: check if a Drive checkpoint exists ──
def checkpoint_exists(name):
    path = os.path.join(DRIVE_PATH, name)
    return os.path.exists(path)

def checkpoint_path(name):
    return os.path.join(DRIVE_PATH, name)

def save_to_drive(src, name):
    dst = os.path.join(DRIVE_PATH, name)
    if os.path.exists(src):
        !cp -r "{src}" "{dst}"
        print(f"  Saved checkpoint: {name}")

def restore_from_drive(name, dst):
    src = os.path.join(DRIVE_PATH, name)
    if os.path.exists(src):
        !cp -r "{src}" "{dst}"
        print(f"  Restored checkpoint: {name} -> {dst}")
        return True
    return False

In [ ]:
#@title === 2. Install Dependencies (~3 min, skips if done) ===
import os, subprocess, atexit
os.environ['QT_QPA_PLATFORM'] = 'offscreen'

# ── Checkpoint: dependencies installed ──
DONE_MARKER = checkpoint_path("deps_installed")
if not os.path.exists(DONE_MARKER):
    print("[1/4] Installing COLMAP + display deps...")
    !apt-get update -qq && apt-get install -y -qq colmap xvfb libgl1-mesa-glx libglib2.0-0
    !colmap version 2>&1 | head -1

    print("[2/4] Installing PyTorch...")
    !pip install torch==2.1.0 torchvision==0.16.0 --index-url https://download.pytorch.org/whl/cu118 -q
    import torch
    cuda_info = f"CUDA: {torch.cuda.is_available()}, VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB" if torch.cuda.is_available() else "CUDA: False"
    print(f"  PyTorch {torch.__version__}, {cuda_info}")

    print("[3/4] Installing Python packages...")
    !pip install plyfile numpy pillow opencv-python-headless tqdm -q

    print("[4/4] Installing 3D Gaussian Splatting...")
    if not os.path.exists('/content/gaussian-splatting'):
        !git clone --recursive https://github.com/graphdeco-inria/gaussian-splatting -q
    %cd /content/gaussian-splatting
    !pip install submodules/diff-gaussian-rasterization -q
    !pip install submodules/simple-knn -q
    import diff_gaussian_rasterization
    import simple_knn
    print("  3DGS rasterizer OK")
    %cd /content

    # Mark complete
    !touch "{DONE_MARKER}"
    print("\nDependencies installed!")
else:
    print("Dependencies already installed. Skipping.")

# ── Virtual display for COLMAP's SiftGPU (always runs) ──
DISPLAY_NUM = 99
os.environ['DISPLAY'] = f':{DISPLAY_NUM}'
LOCK_FILE = f'/tmp/.X{DISPLAY_NUM}-lock'
if not os.path.exists(LOCK_FILE):
    xvfb_proc = subprocess.Popen(
        ['Xvfb', f':{DISPLAY_NUM}', '-screen', '0', '1024x768x24'],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
    )
    atexit.register(lambda: xvfb_proc.terminate())
    print(f"Virtual display :{DISPLAY_NUM} started (PID {xvfb_proc.pid})")
else:
    print(f"Virtual display :{DISPLAY_NUM} already running.")

---
## Step 3: Images (2 min, skips if already downloaded)
---

In [ ]:
#@title === 3: Download Images from GitHub (~2 min, idempotent) ===
import os
import requests

INPUT_DIR = "/content/gaussian-splatting/input"
os.makedirs(INPUT_DIR, exist_ok=True)

# ── Check if already downloaded ──
existing = [f for f in os.listdir(INPUT_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
if len(existing) >= 91:
    print(f"{len(existing)} images already present. Skipping download.")
else:
    GITHUB_REPO = "kaarthik-balakrishnan/arena-3dgs"
    api_url = f"https://api.github.com/repos/{GITHUB_REPO}/contents/splat-files-processed"
    resp = requests.get(api_url)
    if resp.status_code == 200:
        files_list = resp.json()
        for item in files_list:
            if item['name'].lower().endswith(('.jpg', '.jpeg', '.png')):
                img_resp = requests.get(item['download_url'])
                with open(os.path.join(INPUT_DIR, item['name']), 'wb') as f:
                    f.write(img_resp.content)
        imgs = [f for f in os.listdir(INPUT_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        print(f"Downloaded {len(imgs)} images from GitHub")
    else:
        print(f"GitHub API error ({resp.status_code}). Upload manually.")

imgs = [f for f in os.listdir(INPUT_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
print(f"Total: {len(imgs)} images")

---
## COLMAP Step 4: Structure from Motion

Three modular cells: (A) features → (B) matching → (C) reconstruction.
Each saves checkpoints to Drive. If a cell crashes, re-run it.
---

In [ ]:
#@title === 4A: Feature Extraction (~5 min, idempotent) ===
INPUT_DIR = "/content/gaussian-splatting/input"
COLMAP_DIR = "/content/gaussian-splatting/sparse"
os.makedirs(COLMAP_DIR, exist_ok=True)
os.environ['QT_QPA_PLATFORM'] = 'offscreen'

# ── Checkpoint: skip if features already extracted ──
if checkpoint_exists("database.db"):
    print("Database checkpoint found. Restoring...")
    restore_from_drive("database.db", os.path.join(COLMAP_DIR, "database.db"))

DB_PATH = os.path.join(COLMAP_DIR, "database.db")
if os.path.exists(DB_PATH) and os.path.getsize(DB_PATH) > 1000000:
    print("Features already extracted. Skipping.")
else:
    !colmap feature_extractor \
        --database_path {DB_PATH} \
        --image_path {INPUT_DIR} \
        --ImageReader.camera_model SIMPLE_RADIAL \
        --ImageReader.single_camera 1 \
        --SiftExtraction.use_gpu 0 \
        --SiftExtraction.max_num_features 8192 \
        --SiftExtraction.first_octave -1 \
        --SiftExtraction.peak_threshold 0.01
    print("\nSaving checkpoint to Drive...")
    save_to_drive(DB_PATH, "database.db")

# Verify
import sqlite3
conn = sqlite3.connect(DB_PATH)
keypoints = conn.execute("SELECT COUNT(*) FROM keypoints").fetchone()[0]
print(f"  Keypoints tables: {keypoints}")
conn.close()

In [ ]:
#@title === 4B: Feature Matching (~5 min, idempotent) ===
DB_PATH = "/content/gaussian-splatting/sparse/database.db"
os.environ['QT_QPA_PLATFORM'] = 'offscreen'

# ── Checkpoint: skip if matches already computed ──
import sqlite3
conn = sqlite3.connect(DB_PATH)
match_count = conn.execute("SELECT COUNT(*) FROM matches").fetchone()[0]
conn.close()

if match_count > 1000:
    print(f"{match_count} match pairs already exist. Skipping.")
else:
    # Sequential matching (connects consecutive images along walking path)
    print("\n=== Sequential Matching ===")
    !colmap sequential_matcher \
        --database_path {DB_PATH} \
        --SiftMatching.use_gpu 0 \
        --SequentialMatching.overlap 20 \
        --SequentialMatching.max_ratio 0.8 \
        --SequentialMatching.max_distance 0.9
    save_to_drive(DB_PATH, "database.db")

    # Exhaustive matching (global loop closure)
    print("\n=== Exhaustive Matching ===")
    !colmap exhaustive_matcher \
        --database_path {DB_PATH} \
        --SiftMatching.use_gpu 0
    save_to_drive(DB_PATH, "database.db")

# Verify
conn = sqlite3.connect(DB_PATH)
match_count = conn.execute("SELECT COUNT(*) FROM two_view_geometries").fetchone()[0]
print(f"  Verified: {match_count} verified image pairs")
conn.close()

In [ ]:
#@title === 4C: COLMAP Reconstruction (~10 min, idempotent) ===
import struct
INPUT_DIR = "/content/gaussian-splatting/input"
DB_PATH = "/content/gaussian-splatting/sparse/database.db"
COLMAP_DIR = "/content/gaussian-splatting/sparse"
os.environ['QT_QPA_PLATFORM'] = 'offscreen'

# ── Checkpoint: skip if model already exists ──
MODEL_PATH = os.path.join(COLMAP_DIR, "0")
if os.path.exists(os.path.join(MODEL_PATH, "images.bin")):
    with open(os.path.join(MODEL_PATH, "images.bin"), "rb") as f:
        n = struct.unpack("Q", f.read(8))[0]
    print(f"Model already exists ({n} images). Skipping.")
elif checkpoint_exists("sparse_model"):
    print("Restoring sparse model from Drive...")
    # Remove stale dir so cp -r creates it fresh
    import shutil
    if os.path.exists(MODEL_PATH):
        shutil.rmtree(MODEL_PATH)
    restore_from_drive("sparse_model", MODEL_PATH)
else:
    !colmap mapper \
        --database_path {DB_PATH} \
        --image_path {INPUT_DIR} \
        --output_path {COLMAP_DIR} \
        --Mapper.multiple_models 1 \
        --Mapper.max_num_models 50 \
        --Mapper.init_min_tri_angle 4 \
        --Mapper.init_min_num_inliers 15 \
        --Mapper.abs_pose_min_num_inliers 8 \
        --Mapper.ba_local_max_num_iterations 25 \
        --Mapper.ba_global_max_num_iterations 50

# Find best model (largest number of registered images)
best_model = None
best_n = 0
for sub in sorted(os.listdir(COLMAP_DIR)):
    img_path = os.path.join(COLMAP_DIR, sub, "images.bin")
    if os.path.exists(img_path):
        with open(img_path, "rb") as f:
            n = struct.unpack("Q", f.read(8))[0]
        if n > best_n:
            best_n = n
            best_model = sub

if best_model:
    print(f"\nBest model: sub={best_model}, images={best_n}")
    # Save to Drive
    src = os.path.join(COLMAP_DIR, best_model)
    save_to_drive(src, "sparse_model")
else:
    print("No reconstruction produced. Use Step 5 for pre-computed data.")

In [ ]:
#@title === 4D: Model Merging (~2 min, idempotent) ===
# Merges multiple COLMAP reconstructions to get more registered images.
import subprocess

COLMAP_DIR = "/content/gaussian-splatting/sparse"
MERGED_DIR = "/content/gaussian-splatting/sparse_merged"

# ── Checkpoint: skip if already merged ──
if os.path.exists(os.path.join(MERGED_DIR, "images.bin")):
    import struct
    with open(os.path.join(MERGED_DIR, "images.bin"), "rb") as f:
        n = struct.unpack("Q", f.read(8))[0]
    print(f"Merged model already exists ({n} images). Skipping.")
else:
    # Find all reconstructions with at least 5 images
    models = []
    for sub in sorted(os.listdir(COLMAP_DIR)):
        img_path = os.path.join(COLMAP_DIR, sub, "images.bin")
        if os.path.exists(img_path):
            with open(img_path, "rb") as f:
                n = struct.unpack("Q", f.read(8))[0]
            if n >= 5:
                models.append((sub, n))
                print(f"  Found model {sub}: {n} images")

    # If multiple models, try to merge
    if len(models) >= 2:
        print("\nAttempting to merge models...")
        models.sort(key=lambda x: -x[1])
        current = os.path.join(COLMAP_DIR, models[0][0])
        for i in range(1, min(3, len(models))):
            other = os.path.join(COLMAP_DIR, models[i][0])
            merge_out = f"/content/gaussian-splatting/merge_{i}"
            !mkdir -p "{merge_out}"
            result = subprocess.run(
                f"colmap model_merger --input_path1 {current} --input_path2 {other} --output_path {merge_out}",
                shell=True, capture_output=True, text=True
            )
            if os.path.exists(os.path.join(merge_out, "images.bin")):
                with open(os.path.join(merge_out, "images.bin"), "rb") as f:
                    n = struct.unpack("Q", f.read(8))[0]
                print(f"  Merge with {models[i][0]} successful: {n} images")
                current = merge_out
            else:
                print(f"  Merge with {models[i][0]} failed (different coordinate systems)")
                !rm -rf "{merge_out}"
        !cp -r "{current}" "{MERGED_DIR}"

    # Convert best to txt format (for 3DGS)
    best_src = MERGED_DIR if os.path.exists(os.path.join(MERGED_DIR, "images.bin")) else \
               os.path.join(COLMAP_DIR, models[0][0] if models else "0")
    txt_dir = os.path.join(best_src, "txt")
    os.makedirs(txt_dir, exist_ok=True)
    !colmap model_converter --input_path "{best_src}" --output_path "{txt_dir}" --output_type TXT 2>/dev/null

# Report final merged model stats
final = MERGED_DIR if os.path.exists(os.path.join(MERGED_DIR, "images.bin")) else os.path.join(COLMAP_DIR, "0")
if os.path.exists(os.path.join(final, "images.bin")):
    with open(os.path.join(final, "images.bin"), "rb") as f:
        n = struct.unpack("Q", f.read(8))[0]
    with open(os.path.join(final, "points3D.bin"), "rb") as f:
        pts = struct.unpack("Q", f.read(8))[0]
    print(f"\nOptimized COLMAP result: {n} images, {pts} 3D points")
else:
    print("\nNo model available. Use Step 5 to download pre-computed data.")

---
## Step 5: Download Pre-computed COLMAP Data (~1 min)

Skip COLMAP (Step 4) and download pre-computed camera poses.
Choose the optimized 34-image merged model for best results.
---

In [ ]:
#@title === 5: Download Pre-computed COLMAP Data (~1 min, idempotent) ===
import os, requests

INPUT_DIR = "/content/gaussian-splatting/input"
SPARSE_DIR = os.path.join(INPUT_DIR, "sparse", "0")
os.makedirs(SPARSE_DIR, exist_ok=True)
GITHUB_REPO = "kaarthik-balakrishnan/arena-3dgs"
BASE_URL = f"https://raw.githubusercontent.com/{GITHUB_REPO}/main"

# ── Check if already downloaded ──
if os.path.exists(os.path.join(SPARSE_DIR, "images.txt")):
    with open(os.path.join(SPARSE_DIR, "images.txt")) as f:
        n = sum(1 for l in f if l.strip() and not l.startswith('#')) // 2
    print(f"COLMAP data already present ({n} images). Skipping.")
else:
    print("")
    print("Which COLMAP model to download?")
    print("  [1] 34-image merged model (RECOMMENDED — more registered views)")
    print("  [2] 30-image original model")
    choice = input("Enter 1 or 2 (default: 1): ").strip() or "1"

    if choice == "2":
        files_to_download = [
            "colmap_data/cameras.txt",
            "colmap_data/images.txt",
            "colmap_data/points3D.txt",
        ]
        expected_imgs = 30
    else:
        # 34-image merged model stored as optimized sparse data
        files_to_download = [
            "colmap_data_optimized/cameras.txt",
            "colmap_data_optimized/images.txt",
            "colmap_data_optimized/points3D.txt",
        ]
        expected_imgs = 34

    local_names = ["cameras.txt", "images.txt", "points3D.txt"]
    for remote, local in zip(files_to_download, local_names):
        url = f"{BASE_URL}/{remote}"
        print(f"Downloading {local}...")
        r = requests.get(url)
        if r.status_code == 200:
            with open(os.path.join(SPARSE_DIR, local), 'w') as f:
                f.write(r.text)
            print(f"  OK ({len(r.text)/1024:.0f} KB)")
        else:
            print(f"  FAILED (status {r.status_code})")

# Verify
with open(os.path.join(SPARSE_DIR, "images.txt")) as f:
    img_lines = [l for l in f if l.strip() and not l.startswith('#')]
    num_images = len(img_lines) // 2
print(f"\nCOLMAP data: {num_images} registered images")
print(f"  Camera: SIMPLE_RADIAL | Points: ~16K")

---
## Step 6: Convert to 3DGS Format (~1 min)
---

In [ ]:
#@title === 6: Convert Data to 3DGS Format (~1 min, idempotent) ===
%cd /content/gaussian-splatting

import os, glob, subprocess
INPUT_DIR = "/content/gaussian-splatting/input"
SPARSE_DIR = os.path.join(INPUT_DIR, "sparse", "0")
images_dir = os.path.join(INPUT_DIR, "images")

# ── Organize images (idempotent) ──
if os.path.exists(images_dir) and len(os.listdir(images_dir)) >= 30:
    print(f"Images already organized ({len(os.listdir(images_dir))} files).")
else:
    os.makedirs(images_dir, exist_ok=True)
    for ext in ['*.jpg', '*.jpeg', '*.png']:
        for f in glob.glob(os.path.join(INPUT_DIR, ext)):
            os.rename(f, os.path.join(images_dir, os.path.basename(f)))
    print(f"Moved {len(os.listdir(images_dir))} images to input/images/")

# ── Convert COLMAP to PINHOLE + binary (fast, runs every time) ──
required = ["cameras.txt", "images.txt", "points3D.txt"]
missing = [f for f in required if not os.path.exists(os.path.join(SPARSE_DIR, f))]
if missing:
    print(f"ERROR: Missing COLMAP data: {missing}. Run Step 5 first.")
else:
    # Step 1: Convert SIMPLE_RADIAL → PINHOLE in cameras.txt
    # (3DGS only supports undistorted models)
    cam_path = os.path.join(SPARSE_DIR, "cameras.txt")
    with open(cam_path) as f:
        lines = f.readlines()
    modified = False
    with open(cam_path, 'w') as f:
        for line in lines:
            if line.startswith('#') or not line.strip():
                f.write(line)
            else:
                parts = line.strip().split()
                if parts[1] == "SIMPLE_RADIAL":
                    f.write(f"{parts[0]} PINHOLE {parts[2]} {parts[3]} {parts[4]} {parts[4]} {parts[5]} {parts[6]}\n")
                    modified = True
                else:
                    f.write(line)
    if modified:
        print("  Converted camera model: SIMPLE_RADIAL → PINHOLE")

    # Step 2: Always rebuild .bin from .txt (fast, ensures sync)
    for fn in ['cameras.bin', 'images.bin', 'points3D.bin']:
        p = os.path.join(SPARSE_DIR, fn)
        if os.path.exists(p):
            os.remove(p)
    bin_dir = "/content/gaussian-splatting/sparse_bin"
    os.makedirs(bin_dir, exist_ok=True)
    result = subprocess.run(
        f"colmap model_converter --input_path {SPARSE_DIR} --output_path {bin_dir} --output_type BIN",
            shell=True, capture_output=True, text=True
        )
        if os.path.exists(os.path.join(bin_dir, "images.bin")):
            !cp "{bin_dir}/"* "{SPARSE_DIR}/"
            !rm -rf "{bin_dir}"
            print("  Built binary files (cameras.bin, images.bin, points3D.bin)")
        else:
            print(f"WARNING: Binary conversion failed: {result.stderr}")

    # Verify
    with open(os.path.join(SPARSE_DIR, "images.txt")) as f:
        num_images = sum(1 for l in f if l.strip() and not l.startswith('#')) // 2
    print(f"\nReady for training: {num_images} images, PINHOLE model")

---
## Step 7: Train 3D Gaussian Splatting

Training is split into 10-minute segments. Each saves a checkpoint to Drive.
If a segment crashes, re-run it — it will resume from the last checkpoint.

**Run order:**
1. Cell 7A: Quick test (3K iters, ~7 min) — verify everything works
2. Cell 7B: Segment 1 (0→10K iters, ~10 min)
3. Cell 7C: Segment 2 (10K→20K iters, ~10 min)
4. Cell 7D: Segment 3 (20K→30K iters, ~10 min)
5. Cell 7E: Segment 4 (30K→50K iters, ~10 min) — high quality optional
---

In [ ]:
#@title === 7A: Quick Test (3000 iters, ~7 min) ===
%cd /content/gaussian-splatting
import subprocess, sys
INPUT_DIR = "/content/gaussian-splatting/input"
OUTPUT_DIR = "/content/gaussian-splatting/output/quick_test"
os.makedirs(OUTPUT_DIR, exist_ok=True)
PLY_PATH = os.path.join(OUTPUT_DIR, "point_cloud", "iteration_3000", "point_cloud.ply")
if os.path.exists(PLY_PATH):
    print("Quick test already done. Skipping.")
else:
    result = subprocess.run(
        [sys.executable, "train.py",
         "-s", INPUT_DIR,
         "--model_path", OUTPUT_DIR,
         "--iterations", "3000",
         "--test_iterations", "3000",
         "--save_iterations", "3000",
         "--disable_viewer"],
        cwd="/content/gaussian-splatting",
        capture_output=True, text=True
    )
    if result.returncode == 0 and os.path.exists(PLY_PATH):
        print("\nQuick test complete! Proceed to full training.")
    else:
        print(f"\nERROR: Training failed (return code {result.returncode})")
        print("--- STDOUT ---")
        print(result.stdout[-2000:] if len(result.stdout) > 2000 else result.stdout)
        print("--- STDERR ---")
        print(result.stderr[-2000:] if len(result.stderr) > 2000 else result.stderr)

In [ ]:
#@title === 7B: Train Segment 1 (0 → 10K, ~10 min, checkpointed) ===
%cd /content/gaussian-splatting

import subprocess, sys
INPUT_DIR = "/content/gaussian-splatting/input"
OUTPUT_DIR = "/content/gaussian-splatting/output/arena_3dgs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Checkpoint: skip if segment 1 already done ──
chkpt_path = os.path.join(OUTPUT_DIR, "chkpnt10000.pth")
drive_chkpt = checkpoint_path("chkpnt10000.pth")

if os.path.exists(chkpt_path):
    print("Checkpoint at 10K found. Segment 1 already done. Skipping.")
elif os.path.exists(drive_chkpt):
    print("Restoring checkpoint from Drive...")
    !cp "{drive_chkpt}" "{chkpt_path}"
    print("Segment 1 already complete. Skipping.")
else:
    result = subprocess.run(
        [sys.executable, "train.py",
         "-s", INPUT_DIR,
         "--model_path", OUTPUT_DIR,
         "--iterations", "10000",
         "--checkpoint_iterations", "10000",
         "--test_iterations", "10000",
         "--save_iterations", "10000",
         "--disable_viewer"],
        cwd="/content/gaussian-splatting",
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f"\nERROR: Training crashed (code {result.returncode})")
        print("--- STDERR ---")
        print(result.stderr[-2000:] if len(result.stderr) > 2000 else result.stderr)
    elif os.path.exists(chkpt_path):
        save_to_drive(chkpt_path, "chkpnt10000.pth")
        print("\nSegment 1 complete! Checkpoint saved to Drive.")
    else:
        print("\nWARNING: Checkpoint not found. Training may have failed.")

In [ ]:
#@title === 7C: Train Segment 2 (10K → 20K, ~10 min, checkpointed) ===
%cd /content/gaussian-splatting

import subprocess, sys
INPUT_DIR = "/content/gaussian-splatting/input"
OUTPUT_DIR = "/content/gaussian-splatting/output/arena_3dgs"
START_CKPT = os.path.join(OUTPUT_DIR, "chkpnt10000.pth")
TARGET_CKPT = os.path.join(OUTPUT_DIR, "chkpnt20000.pth")
DRIVE_CKPT = checkpoint_path("chkpnt20000.pth")

# ── Checkpoint: skip if segment 2 already done ──
if os.path.exists(TARGET_CKPT):
    print("Checkpoint at 20K found. Segment 2 already done. Skipping.")
elif os.path.exists(DRIVE_CKPT):
    print("Restoring checkpoint from Drive...")
    !cp "{DRIVE_CKPT}" "{TARGET_CKPT}"
    print("Segment 2 already complete. Skipping.")
elif not os.path.exists(START_CKPT):
    print("Need checkpoint from Segment 1 (10K). Restoring from Drive...")
    ok = restore_from_drive("chkpnt10000.pth", START_CKPT)
    if not ok:
        raise RuntimeError("10K checkpoint not found on Drive. Run Cell 7B first.")

if os.path.exists(START_CKPT) and not os.path.exists(TARGET_CKPT):
    result = subprocess.run(
        [sys.executable, "train.py",
         "-s", INPUT_DIR,
         "--model_path", OUTPUT_DIR,
         "--start_checkpoint", START_CKPT,
         "--iterations", "20000",
         "--checkpoint_iterations", "15000", "20000",
         "--test_iterations", "20000",
         "--save_iterations", "20000",
         "--densify_until_iter", "15000",
         "--disable_viewer"],
        cwd="/content/gaussian-splatting",
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f"\nERROR: Training crashed (code {result.returncode})")
        print("--- STDERR ---")
        print(result.stderr[-2000:] if len(result.stderr) > 2000 else result.stderr)
    elif os.path.exists(TARGET_CKPT):
        save_to_drive(TARGET_CKPT, "chkpnt20000.pth")
        print("\nSegment 2 complete! Checkpoint saved to Drive.")
    else:
        print("\nWARNING: Checkpoint not found.")

In [ ]:
#@title === 7D: Train Segment 3 (20K → 30K, ~10 min, checkpointed) ===
%cd /content/gaussian-splatting

import subprocess, sys
INPUT_DIR = "/content/gaussian-splatting/input"
OUTPUT_DIR = "/content/gaussian-splatting/output/arena_3dgs"
START_CKPT = os.path.join(OUTPUT_DIR, "chkpnt20000.pth")
TARGET_CKPT = os.path.join(OUTPUT_DIR, "chkpnt30000.pth")
DRIVE_CKPT = checkpoint_path("chkpnt30000.pth")

# ── Checkpoint: skip if segment 3 already done ──
if os.path.exists(TARGET_CKPT):
    print("Checkpoint at 30K found. Segment 3 already done. Skipping.")
elif os.path.exists(DRIVE_CKPT):
    print("Restoring checkpoint from Drive...")
    !cp "{DRIVE_CKPT}" "{TARGET_CKPT}"
    print("Segment 3 already complete. Skipping.")
elif not os.path.exists(START_CKPT):
    print("Need 20K checkpoint. Restoring from Drive...")
    ok = restore_from_drive("chkpnt20000.pth", START_CKPT)
    if not ok:
        raise RuntimeError("20K checkpoint not found on Drive. Run Cell 7C first.")

if os.path.exists(START_CKPT) and not os.path.exists(TARGET_CKPT):
    result = subprocess.run(
        [sys.executable, "train.py",
         "-s", INPUT_DIR,
         "--model_path", OUTPUT_DIR,
         "--start_checkpoint", START_CKPT,
         "--iterations", "30000",
         "--checkpoint_iterations", "25000", "30000",
         "--test_iterations", "30000",
         "--save_iterations", "30000",
         "--position_lr_max_steps", "30000",
         "--disable_viewer"],
        cwd="/content/gaussian-splatting",
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f"\nERROR: Training crashed (code {result.returncode})")
        print("--- STDERR ---")
        print(result.stderr[-2000:] if len(result.stderr) > 2000 else result.stderr)
    elif os.path.exists(TARGET_CKPT):
        save_to_drive(TARGET_CKPT, "chkpnt30000.pth")
        print("\nSegment 3 complete! Checkpoint saved to Drive.")
    else:
        print("\nWARNING: Checkpoint not found.")

In [ ]:
#@title === 7E (Optional): High Quality Extension 30K → 50K (~10 min) ===
%cd /content/gaussian-splatting

import subprocess, sys
INPUT_DIR = "/content/gaussian-splatting/input"
OUTPUT_DIR = "/content/gaussian-splatting/output/arena_3dgs"
START_CKPT = os.path.join(OUTPUT_DIR, "chkpnt30000.pth")
TARGET_CKPT = os.path.join(OUTPUT_DIR, "chkpnt50000.pth")
DRIVE_CKPT = checkpoint_path("chkpnt50000.pth")

if os.path.exists(TARGET_CKPT):
    print("50K checkpoint found. Skipping.")
elif os.path.exists(DRIVE_CKPT):
    print("Restoring 50K from Drive...")
    !cp "{DRIVE_CKPT}" "{TARGET_CKPT}"
elif not os.path.exists(START_CKPT):
    print("Need 30K checkpoint. Restoring from Drive...")
    ok = restore_from_drive("chkpnt30000.pth", START_CKPT)
    if not ok:
        raise RuntimeError("30K checkpoint not found on Drive. Run Cells 7B-7D first.")

if os.path.exists(START_CKPT) and not os.path.exists(TARGET_CKPT):
    print("Training 30K → 50K for maximum quality...")
    result = subprocess.run(
        [sys.executable, "train.py",
         "-s", INPUT_DIR,
         "--model_path", OUTPUT_DIR,
         "--start_checkpoint", START_CKPT,
         "--iterations", "50000",
         "--checkpoint_iterations", "40000", "45000", "50000",
         "--test_iterations", "50000",
         "--save_iterations", "50000",
         "--position_lr_max_steps", "50000",
         "--disable_viewer"],
        cwd="/content/gaussian-splatting",
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f"\nERROR: Training crashed (code {result.returncode})")
        print("--- STDERR ---")
        print(result.stderr[-2000:] if len(result.stderr) > 2000 else result.stderr)
    elif os.path.exists(TARGET_CKPT):
        save_to_drive(TARGET_CKPT, "chkpnt50000.pth")
        print("\n50K training complete! Checkpoint saved to Drive.")
    else:
        print("\nWARNING: Checkpoint not found.")

---
## Step 8: Export & Validate for Unity (~1 min)
---

In [ ]:
#@title === 8A: Export Point Cloud (~1 min) ===
import os
from google.colab import files

# Find latest trained model
OUTPUT_DIR = "/content/gaussian-splatting/output"
PLY_DST = "/content/arena_3dgs_pointcloud.ply"

# Check various possible output locations
candidates = [
    os.path.join(OUTPUT_DIR, "arena_3dgs", "point_cloud"),
    os.path.join(OUTPUT_DIR, "point_cloud"),
]

found_ply = None
for base in candidates:
    if os.path.exists(base):
        iters = sorted([d for d in os.listdir(base) if d.startswith("iteration_")])
        if iters:
            latest = os.path.join(base, iters[-1], "point_cloud.ply")
            if os.path.exists(latest):
                found_ply = latest
                print(f"Found: {latest}")
                break

if found_ply:
    !cp "{found_ply}" "{PLY_DST}"
    size_mb = os.path.getsize(PLY_DST) / (1024 * 1024)
    print(f"\nPoint cloud: {PLY_DST} ({size_mb:.1f} MB)")

    # Backup to Drive
    save_to_drive(PLY_DST, "arena_3dgs_pointcloud.ply")
    print(f"Backed up to Drive: {checkpoint_path('arena_3dgs_pointcloud.ply')}")
    print(f"\nDownloading to your computer...")
    files.download(PLY_DST)
else:
    print("No trained model found. Run training cells (7A-7E) first.")

In [ ]:
#@title === 8B: Validate PLY for Unity (~1 min) ===
# Check that the PLY has the right format for UnityGaussianSplatting
PLY_PATH = "/content/arena_3dgs_pointcloud.ply"

if not os.path.exists(PLY_PATH):
    print("No PLY found. Run 8A first.")
else:
    from plyfile import PlyData
    import numpy as np

    ply = PlyData.read(PLY_PATH)
    data = ply['vertex'].data
    n = len(data)
    print(f"Gaussians: {n:,}")

    # Check UnityGaussianSplatting required properties
    required = ['x', 'y', 'z', 'f_dc_0', 'f_dc_1', 'f_dc_2',
                'opacity', 'scale_0', 'scale_1', 'scale_2',
                'rot_0', 'rot_1', 'rot_2', 'rot_3']
    missing = [p for p in required if p not in data.dtype.names]
    if missing:
        print(f"\nWARNING: Unity requires: {missing}")
    else:
        print(f"\nUnity format: OK")

    # Stats
    xyz = np.stack([data['x'], data['y'], data['z']], axis=1)
    print(f"Bounds: X[{xyz[:,0].min():.1f}, {xyz[:,0].max():.1f}] "
          f"Y[{xyz[:,1].min():.1f}, {xyz[:,1].max():.1f}]")

    opacities = data['opacity']
    visible = (np.array(opacities) > 0.01).sum()
    print(f"Visible Gaussians (>0.01 opacity): {visible:,}")
    print(f"File size: {os.path.getsize(PLY_PATH) / 1024**2:.1f} MB")

print("\n" + "=" * 50)
print("  VIEWING OPTIONS")
print("=" * 50)
print("\n1. SuperSplat (no install, web):")
print("     https://supersplat.com/")
print("\n2. Unity walkthrough (best):")
print("     Clone: https://github.com/aras-p/UnityGaussianSplatting")
print("     Drop PLY into Assets/GaussianAssets/")
print("     WASD + mouse-look controls")

---
## Appendix: Troubleshooting

| Problem | Solution |
|---------|----------|
| **Session disconnected** | All checkpoints saved to Drive. Just re-run the cell that failed. |
| **CUDA out of memory** | Skip 50K cell (7E). Use 30K model. |
| **COLMAP produces 0 images** | Download pre-computed data (Step 5). |
| **Dropbox restore failed** | Check `MyDrive/arena_3dgs/` exists and has files. |
| **Training segment needs re-run** | It will resume from the last checkpoint automatically. |

**Checkpoint locations (in Google Drive):**
- `MyDrive/arena_3dgs/database.db` — COLMAP features + matches
- `MyDrive/arena_3dgs/sparse_model` — COLMAP reconstruction
- `MyDrive/arena_3dgs/chkpnt*.pth` — Training checkpoints at 10K, 20K, 30K, 50K
- `MyDrive/arena_3dgs/arena_3dgs_pointcloud.ply` — Final PLY export

---